# Bengaluru Area-Risk Model — Training (M3)

Spatiotemporal risk-probability model for tourist safety.

India has **no public point-level crime data**, so we use real NCRB / data.gov.in
aggregates as **zone-level priors** and generate synthetic geocoded + timestamped
incidents conditioned on them, then train a classical gradient-boosted classifier.

**Colab:** run the install cell, then clone/upload the `app/` package (or `pip install`
the project) so the imports below resolve.

In [ ]:
# Colab only: install deps and make the project importable.
# !pip install scikit-learn pandas numpy joblib
# !git clone <your-repo-url> && cd tourist-safety  # then add to sys.path
import sys, os
sys.path.append(os.path.abspath('..'))  # repo root, so `import app...` works

## 1. Inspect the Bengaluru priors & temporal patterns

In [ ]:
from app.ml.bengaluru_priors import ZONE_PRIORS, spatial_intensity, temporal_multiplier

for z in ZONE_PRIORS:
    print(f"{z.name:30s} intensity={z.intensity}")

print('\nMajestic intensity :', round(spatial_intensity(12.9770, 77.5720), 3))
print('Cubbon Park intensity:', round(spatial_intensity(12.9760, 77.5950), 3))
print('Sat 22:00 vs Tue 10:00:', round(temporal_multiplier(22, 5), 2), 'vs', round(temporal_multiplier(10, 1), 2))

## 2. Build the synthetic spatiotemporal dataset

In [ ]:
from app.ml.dataset import build_dataset

data = build_dataset(n_days=90, n_samples=60_000, seed=42)
print('positive rate:', round(data.meta['positive_rate'], 3))
data.frame.head()

## 3. Train & evaluate

In [ ]:
from app.ml.risk_model import train_model, save_bundle

result = train_model(data, seed=42)
result.metrics

## 4. Inference: risk score for (location, time)

In [ ]:
from datetime import datetime, UTC
from app.ml.risk_model import predict_risk

b = result.bundle
print('Majestic, Sat 22:00 :', round(predict_risk(12.9770, 77.5720, datetime(2026,6,20,22,tzinfo=UTC), bundle=b), 3))
print('Majestic, Tue 10:00 :', round(predict_risk(12.9770, 77.5720, datetime(2026,6,16,10,tzinfo=UTC), bundle=b), 3))
print('Cubbon Park, Tue 10:00:', round(predict_risk(12.9760, 77.5950, datetime(2026,6,16,10,tzinfo=UTC), bundle=b), 3))

# Persist for serving
# save_bundle(result.bundle, '../models/area_risk_model.joblib')